<a href="https://colab.research.google.com/github/Bunnydragon96/Shortest-Path-Route-CIIC-4070/blob/main/Project3_ShortestPathRouting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [CIIC4070/ICOM5026] Computer Networks
## Project 3 – Shortest Path Routing

**University of Puerto Rico – Mayagüez Campus**  

---

## 1. Network Definition

The network is modelled as a **weighted undirected graph** with 14 nodes representing U.S. geographic regions and edges weighted by link cost (in miles). The graph was provided in the project specification.

| Node ID | Label | Node ID | Label |
|---------|-------|---------|-------|
| 1 | WA | 8 | IL |
| 2 | CA1 | 9 | PA |
| 3 | CA2 | 10 | GA |
| 4 | UT | 11 | MI |
| 5 | CO | 12 | NY |
| 6 | TX | 13 | NJ |
| 7 | NE | 14 | DC |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import heapq
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from collections import defaultdict

# ── Network adjacency list (undirected, weights = link cost in miles) ─────────
EDGES = [
    ('WA',  'CA1', 1050),
    ('WA',  'UT',  1500),
    ('WA',  'NE',  2400),
    ('WA',  'IL',  1950),
    ('CA1', 'CA2',  600),
    ('CA1', 'UT',   750),
    ('CA2', 'TX',  1800),
    ('UT',  'CO',   600),
    ('CO',  'NE',   600),
    ('CO',  'TX',  1200),
    ('NE',  'IL',   750),
    ('NE',  'TX',  1350),
    ('IL',  'PA',   750),
    ('TX',  'PA',  1800),
    ('TX',  'GA',  1050),
    ('PA',  'MI',   300),
    ('PA',  'NY',   300),
    ('PA',  'NJ',   300),
    ('PA',  'DC',   750),
    ('MI',  'NY',   600),
    ('NY',  'NJ',   300),
    ('NJ',  'DC',   150),
    ('GA',  'DC',   750),
]

# Build adjacency dict
def build_graph(edges):
    graph = defaultdict(list)
    for u, v, w in edges:
        graph[u].append((v, w))
        graph[v].append((u, w))
    return dict(graph)

graph = build_graph(EDGES)
nodes = sorted(graph.keys())
print(f'Nodes ({len(nodes)}): {nodes}')
print(f'Edges: {len(EDGES)}')

## 2. Graph Visualisation

In [ ]:
# ── Fixed positions reflecting geographic layout ──────────────────────────────
POS = {
    'WA':  (0.5, 9.0), 'CA1': (0.5, 7.0), 'CA2': (1.0, 5.5),
    'UT':  (2.0, 7.0), 'CO':  (3.0, 6.5), 'NE':  (4.5, 7.5),
    'TX':  (4.0, 4.5), 'IL':  (6.0, 7.5), 'PA':  (8.0, 7.0),
    'GA':  (8.0, 4.0), 'DC':  (9.2, 5.5), 'MI':  (8.0, 8.5),
    'NY':  (9.5, 8.0), 'NJ':  (9.5, 6.5),
}

G = nx.Graph()
for u, v, w in EDGES:
    G.add_edge(u, v, weight=w)

fig, ax = plt.subplots(figsize=(14, 8))
nx.draw_networkx_nodes(G, POS, node_size=700, node_color='#4A90D9', ax=ax)
nx.draw_networkx_labels(G, POS, font_size=9, font_color='white', font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, POS, width=1.5, edge_color='#555', ax=ax)
edge_labels = {(u, v): w for u, v, w in EDGES}
nx.draw_networkx_edge_labels(G, POS, edge_labels=edge_labels, font_size=7, ax=ax)
ax.set_title('Computer Network – 14 Nodes (link costs in miles)', fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('network_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graph saved to network_graph.png')

## 3. Dijkstra's Algorithm

### 3.1 Pseudocode

```
DIJKSTRA(Graph G, source s, destination t):
  1.  dist[v] ← ∞  for all v ∈ V                  // initialise distances
  2.  prev[v] ← None  for all v ∈ V                // initialise predecessors
  3.  dist[s] ← 0
  4.  Q ← min-priority queue containing all v with priority dist[v]
  5.  while Q is not empty:
  6.      u ← EXTRACT-MIN(Q)                        // O(log V) with binary heap
  7.      if u == t: break                           // destination reached
  8.      for each neighbour v of u with edge weight w:
  9.          alt ← dist[u] + w
  10.         if alt < dist[v]:
  11.             dist[v] ← alt
  12.             prev[v] ← u
  13.             DECREASE-KEY(Q, v, alt)            // re-insert with new priority
  14. return reconstruct_path(prev, s, t), dist[t]
```

### 3.2 Complexity Analysis

| Component | Binary Heap | Fibonacci Heap |
|-----------|------------|----------------|
| EXTRACT-MIN | O(log V) × V = **O(V log V)** | O(log V) amortised |
| DECREASE-KEY | O(log V) × E = **O(E log V)** | O(1) amortised |
| **Total** | **O((V + E) log V)** | **O(V log V + E)** |

For a sparse graph (E ≈ V): **O(V log V)**. For a dense graph (E ≈ V²): **O(V² log V)**.
With a binary heap (as implemented below) the overall complexity is **O((V + E) log V)**.

In [ ]:
# ── Dijkstra's Algorithm Implementation ──────────────────────────────────────
def dijkstra(graph, source, destination):
    """
    Dijkstra's shortest-path algorithm.

    Parameters
    ----------
    graph       : dict  adjacency list {node: [(neighbour, weight), ...]}
    source      : str   starting node
    destination : str   target node

    Returns
    -------
    path : list[str]  ordered sequence of nodes on the shortest path
    cost : int/float  total path cost; math.inf if unreachable
    """
    # Initialise distances and predecessors
    dist = {v: math.inf for v in graph}
    prev = {v: None for v in graph}
    dist[source] = 0

    # Min-heap: (cost, node)
    heap = [(0, source)]
    visited = set()

    while heap:
        cost_u, u = heapq.heappop(heap)

        if u in visited:
            continue
        visited.add(u)

        if u == destination:
            break

        for v, w in graph.get(u, []):
            alt = cost_u + w
            if alt < dist[v]:
                dist[v] = alt
                prev[v] = u
                heapq.heappush(heap, (alt, v))

    # Reconstruct path
    path = []
    node = destination
    while node is not None:
        path.append(node)
        node = prev[node]
    path.reverse()

    if path[0] != source:
        return [], math.inf  # no path found

    return path, dist[destination]

print('dijkstra() loaded successfully.')

## 4. Test Cases

The three test cases required by the project specification are run below.

In [ ]:
# ── Helper: pretty-print result ───────────────────────────────────────────────
def print_result(src, dst, path, cost):
    arrow = ' → '.join(path)
    print(f'  Source      : {src}')
    print(f'  Destination : {dst}')
    print(f'  Path        : {arrow}')
    print(f'  Total cost  : {cost:,} miles')
    print()

# ── Helper: highlight path on graph ──────────────────────────────────────────
def plot_path(G, pos, path, title):
    path_edges = list(zip(path, path[1:]))
    path_nodes = set(path)

    node_colors = ['#E74C3C' if n in path_nodes else '#4A90D9' for n in G.nodes()]
    edge_colors = ['#E74C3C' if (u,v) in path_edges or (v,u) in path_edges else '#AAAAAA'
                   for u, v in G.edges()]
    edge_widths = [3.0 if (u,v) in path_edges or (v,u) in path_edges else 1.0
                   for u, v in G.edges()]

    fig, ax = plt.subplots(figsize=(14, 7))
    nx.draw_networkx_nodes(G, pos, node_size=700, node_color=node_colors, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=9, font_color='white', font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color=edge_colors, ax=ax)
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, ax=ax)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
    red_patch   = mpatches.Patch(color='#E74C3C', label='Shortest path')
    blue_patch  = mpatches.Patch(color='#4A90D9', label='Other nodes/edges')
    ax.legend(handles=[red_patch, blue_patch], loc='lower right')
    plt.tight_layout()
    plt.show()

print('Helpers loaded.')

In [ ]:
# ── Test Case 1: NY → CA2 ────────────────────────────────────────────────────
print('=' * 60)
print('TEST CASE 1: Shortest path from NY to CA2')
print('=' * 60)
path1, cost1 = dijkstra(graph, 'NY', 'CA2')
print_result('NY', 'CA2', path1, cost1)
plot_path(G, POS, path1, f'Test Case 1 – NY → CA2  (cost: {cost1:,} miles)')

In [ ]:
# ── Test Case 2: WA → GA ────────────────────────────────────────────────────
print('=' * 60)
print('TEST CASE 2: Shortest path from WA to GA')
print('=' * 60)
path2, cost2 = dijkstra(graph, 'WA', 'GA')
print_result('WA', 'GA', path2, cost2)
plot_path(G, POS, path2, f'Test Case 2 – WA → GA  (cost: {cost2:,} miles)')

In [ ]:
# ── Test Case 3: CA1 → NJ ────────────────────────────────────────────────────
print('=' * 60)
print('TEST CASE 3: Shortest path from CA1 to NJ')
print('=' * 60)
path3, cost3 = dijkstra(graph, 'CA1', 'NJ')
print_result('CA1', 'NJ', path3, cost3)
plot_path(G, POS, path3, f'Test Case 3 – CA1 → NJ  (cost: {cost3:,} miles)')

## 5. All-Pairs Shortest Paths (Floyd-Warshall)

For completeness, we also implement **Floyd-Warshall** to compute all-pairs shortest paths.

**Complexity:** O(V³) time, O(V²) space.

In [ ]:
# ── Floyd-Warshall Implementation ────────────────────────────────────────────
def floyd_warshall(nodes, edges):
    """
    Floyd-Warshall all-pairs shortest path.
    Returns dist matrix and next-hop matrix.
    """
    idx  = {n: i for i, n in enumerate(nodes)}
    n    = len(nodes)
    dist = [[math.inf]*n for _ in range(n)]
    nxt  = [[None]*n for _ in range(n)]

    for i in range(n):
        dist[i][i] = 0

    for u, v, w in edges:
        i, j = idx[u], idx[v]
        dist[i][j] = w; nxt[i][j] = v
        dist[j][i] = w; nxt[j][i] = u

    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
                    nxt[i][j]  = nxt[i][k]
    return dist, nxt, idx

def fw_path(nxt, idx, src, dst):
    nodes_list = list(idx.keys())
    if nxt[idx[src]][idx[dst]] is None:
        return []
    path = [src]
    while path[-1] != dst:
        path.append(nxt[idx[path[-1]]][idx[dst]])
    return path

fw_dist, fw_nxt, fw_idx = floyd_warshall(nodes, EDGES)

print('Floyd-Warshall results for the three test cases:')
print()
for src, dst in [('NY', 'CA2'), ('WA', 'GA'), ('CA1', 'NJ')]:
    p = fw_path(fw_nxt, fw_idx, src, dst)
    c = fw_dist[fw_idx[src]][fw_idx[dst]]
    print(f'  {src} → {dst}: {" → ".join(p)}  [cost: {c:,}]')

## 6. Algorithm Comparison

| Property | Dijkstra | Floyd-Warshall | Bellman-Ford | A* |
|----------|----------|---------------|-------------|----|
| Time complexity | O((V+E) log V) | O(V³) | O(VE) | O(E log V)* |
| Space complexity | O(V) | O(V²) | O(V) | O(V) |
| Negative weights | ✗ | ✓ | ✓ | ✗ |
| All pairs | ✗ | ✓ | ✗ | ✗ |
| Heuristic | ✗ | ✗ | ✗ | ✓ |
| Best use case | Single-source, non-negative | All-pairs | Negative weights | Spatial graphs |

*A* with an admissible heuristic.

**Choice:** Dijkstra's algorithm was selected for this project because:
1. All edge weights are non-negative (link costs ≥ 0).
2. We need single-source, single-destination paths.
3. It is computationally efficient for sparse graphs: O((V+E) log V).
4. It is the basis of many real-world routing protocols (e.g., OSPF uses a variant of Dijkstra).

## 7. Summary

| Test Case | Source | Destination | Shortest Path | Cost (miles) |
|-----------|--------|-------------|---------------|--------------|
| 1 | NY | CA2 | See output above | See above |
| 2 | WA | GA | See output above | See above |
| 3 | CA1 | NJ | See output above | See above |

Run all cells to populate the table with computed values.